# CCTV Candidate Evaluation Protocol

## Objective

CCTV 사람 속성·식별 후보를 같은 조건에서 비교하되, proxy 점수를 production 성과로 바꾸지 않는다. manifest·근거 파일·독립 identity 라벨·track-heldout·사람 검토를 모두 확인한 결과만 다음 단계로 보낸다.


In [1]:
from pathlib import Path
import json
import sys

project_root = Path.cwd()
if not (project_root / 'configs').is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / 'src'))

from cctv_eval_harness.gate import evaluate

{'workspace': '.', 'runtime': 'ready'}


{'workspace': '.', 'runtime': 'ready'}

## Experiment contract

- Baseline: 후보별 동일 입력·동일 스키마·동일 지연시간 정의
- Comparison: 속성 점수만이 아니라 identity·track-heldout·false-match·review rate를 함께 본다.
- Gate: 독립 라벨, provenance, 사람 검토가 없으면 점수와 무관하게 보류한다.
- Scope: 이 공개판의 예제는 합성 proxy 기록이며 CCTV 영상·가중치·원본 라벨을 포함하지 않는다.


In [2]:
payload = json.loads((project_root / 'examples' / 'proxy_result.json').read_text(encoding='utf-8'))
policy = json.loads((project_root / 'configs' / 'promotion_gate.json').read_text(encoding='utf-8'))
report = evaluate(payload, policy, project_root)
report


{'schemaVersion': 'cctv-candidate-promotion-report-v1',
 'candidate': 'clip-vit-l14-proxy',
 'status': 'NOT_APPROVED',
 'reasons': ['measurement is not a sealed identity and track-heldout evaluation',
  'independent identity labels are unavailable',
  'track-heldout metrics are not eligible',
  'human review is incomplete',
  'manifest and evaluation evidence references are missing',
  'attributeMacroF1 is below threshold',
  'identityRank1 is missing',
  'identityRecallAt5 is missing',
  'falseMatchRate is missing']}

## Decision

이 예제는 `NOT_APPROVED`가 정답이다. proxy 속성 결과가 있어도 독립 identity 라벨, track-heldout, 사람 검토, 검증 가능한 산출물 참조가 없기 때문이다. 이 상태를 실패로 숨기지 않고 다음 데이터·split·검토 작업으로 환류한다.


In [3]:
assert report['status'] == 'NOT_APPROVED'
{'candidate': report['candidate'], 'status': report['status'], 'reason_count': len(report['reasons'])}


{'candidate': 'clip-vit-l14-proxy',
 'status': 'NOT_APPROVED',
 'reason_count': 9}

## Next steps

1. 권한 있는 실제 CCTV 데이터에서 identity·camera·track 기준으로 split을 고정한다.
2. manifest·결과 JSON·source hash를 묶어 재현한다.
3. 사람 검토와 independent ground truth를 확보한 뒤에만 sealed identity gate를 다시 실행한다.
